# 04 — Hillstrom: advanced learners, calibration and the benchmark (Day 3)

Notebook 03 found the uplift signal on `visit` weak and the response baseline
embarrassingly competitive. This notebook pushes harder on three fronts and
then picks a **champion**:

1. **Stronger learners** — the DR-learner and the causal forest, which use the
  data differently from S/T/X and (critically) report **effect intervals**.
2. **Calibration** — do the numbers mean what they say? Two distinct questions:
  outcome-model calibration (per arm, Brier + reliability) and CATE calibration
  (predicted vs observed uplift by decile). Plus a recalibration decision.
3. **Stability** — repeat everything over **10 seeds** and report mean ± spread,
  because a 0.003 Qini gap on one split is noise, not a result.

Vocabulary lives in [`docs/concepts.md`](../../docs/concepts.md). New here:
**DR-learner**, **causal forest**, **effect interval**, **Brier score**,
**reliability curve**, **recalibration**, **CATE calibration**.

| Step | Question | Tool |
| --- | --- | --- |
| Advanced learners | Do DR / forest beat S/T/X? | `packages/causal/learners.py` |
| Effect intervals | How certain is each customer's effect? | `effect_interval` |
| CATE calibration | Do predicted magnitudes match observed uplift? | decile table |
| Outcome calibration | Are the arm probabilities right? | Brier + reliability |
| Recalibration | Does isotonic or Platt help? | fit on **validation** only |
| Stability | Is any edge real? | 10-seed benchmark |

**The answer, up front:** the causal forest and DR-learner join the pack but do
not break away on `visit`; calibration shows the models **rank** better than
their **magnitudes** deserve; and the 10-seed benchmark decides the champion.
Sections 6–7 carry the numbers and the write-up.

## 0. Setup

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")  # causalml/lightgbm/econml emit benign warnings

from packages.causal.benchmark import BusinessAssumptions, run_benchmark
from packages.causal.calibration import (
    cate_calibration_by_decile,
    fit_recalibrator,
    reliability_by_arm,
)
from packages.causal.features import build_features, encode_splits, make_splits
from packages.causal.learners import make_learner
from packages.causal.metrics import random_uplift, ranking_table, uplift_at_k
from packages.causal.plotting import plot_qini
from packages.datasets.hillstrom import fetch_hillstrom

pd.set_option("display.width", 140)
RANDOM_STATE = 42
print("repo root:", ROOT)

## 1. Load and split

Same `mens` arm, same stratified `treatment × visit` split as notebook 03, so
the Day-2 and Day-3 numbers are directly comparable. The **validation** split
— which notebook 03 left unused — now earns its place: it is where
recalibration gets fitted (§5), keeping the test split untouched until the
final number is read.

In [ ]:
df = fetch_hillstrom("mens")
X, y, t = build_features(df, target="visit")
splits, feature_names = encode_splits(make_splits(X, y, t, random_state=RANDOM_STATE))
X_tr, y_tr, t_tr = splits["train"]
X_val, y_val, t_val = splits["val"]
X_te, y_te, t_te = splits["test"]

Xv, tv, yv = X_tr.values.astype(float), t_tr.values.astype(float), y_tr.values.astype(float)
Xvalf, yvalf, tvalf = (
    X_val.values.astype(float), y_val.values.astype(float), t_val.values.astype(float)
)
Xtef, ytef, ttef = X_te.values.astype(float), y_te.values.astype(float), t_te.values.astype(float)

print(f"rows: train {len(Xv):,}  val {len(Xvalf):,}  test {len(Xtef):,}")
print(f"ATE on test (visit): {ytef[ttef == 1].mean() - ytef[ttef == 0].mean():+.4f}")

## 2. DR-learner and causal forest on `visit`

Both estimators come from `econml` via the `make_learner` factory, and both
use the data more efficiently than the T/X imputation step:

| Learner | Idea | Why it might help here |
| --- | --- | --- |
| **DR-learner** | Regress *doubly robust* pseudo-outcomes on features | Combines an outcome model and a propensity model; lower variance than imputing |
| **causal forest** | Honest, orthogonalised random forest over small leaf effects | Built for heterogeneity; reports **asymptotic effect intervals** for free |

The factory handles both libraries' quirks (`fit(Y, T, X=X)`, and
`effect_interval` returning `(lower, upper)` *without* the point estimate),
so the notebook stays readable.

In [ ]:
scores = {}

# Day-2 reference points (same config as notebook 03) for a fair comparison.
for name in ("response", "s", "t", "x"):
    m = make_learner(name, target="visit", random_state=RANDOM_STATE)
    m.fit(Xv, tv, yv)
    scores[name] = m.predict(Xtef)

# Day-3 challengers.
dr_model = make_learner("dr", target="visit", random_state=RANDOM_STATE)
dr_model.fit(Xv, tv, yv)
scores["dr"] = dr_model.predict(Xtef)

forest_model = make_learner("forest", target="visit", random_state=RANDOM_STATE)
forest_model.fit(Xv, tv, yv)
scores["forest"] = forest_model.predict(Xtef)

scores["random"] = random_uplift(ytef, ttef, random_state=RANDOM_STATE)

table = ranking_table(ytef, ttef, scores)
table

In [ ]:
fig = plot_qini(
    ytef, ttef, scores,
    title="Qini curves — Day-3 learners vs Day-2 pack (visit)",
    name="day3_qini_dr_forest_vs_pack",
)
plt.show()

![Qini curves — Day-3 learners vs Day-2 pack](../../docs/figures/day3_qini_dr_forest_vs_pack.png)

## 3. Effect intervals — how certain is each customer's effect?

This is the capability the Day-4 optimizer actually needs: a profit number is
only trustworthy if the *effect* underneath it has an interval. The causal
forest gives asymptotic 95% intervals per customer.

**The honest reading** is about *coverage*, not the point estimates: the
fraction of intervals that **exclude zero** tells us how many customers we can
confidently label persuadable (or lost causes) at all.

In [ ]:
point = scores["forest"]
lower, upper = forest_model.effect_interval(Xtef, alpha=0.05)
width = upper - lower

print(f"point estimate range : [{point.min():.4f}, {point.max():.4f}]")
print(f"mean 95% CI width    : {width.mean():.4f}")
print(f"ATE on test          : {ytef[ttef == 1].mean() - ytef[ttef == 0].mean():.4f}")
print(f"fraction of point estimates > 0          : {(point > 0).mean():.3f}")
print(f"fraction of CIs excluding 0 (either side): {((lower > 0) | (upper < 0)).mean():.3f}")

# Visualise a slice: 40 customers around the median score, with intervals.
order = np.argsort(point)
pick = order[np.linspace(0, len(order) - 1, 40).astype(int)]
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(pick))
ax.errorbar(
    x_pos, point[pick],
    yerr=[point[pick] - lower[pick], upper[pick] - point[pick]],
    fmt="o", ms=3, lw=1, capsize=2, color="#64b5cd", ecolor="#999999",
)
ax.axhline(0, color="#c44e52", lw=1, ls="--", label="zero effect")
ate_test = ytef[ttef == 1].mean() - ytef[ttef == 0].mean()
ax.axhline(ate_test, color="#333333", lw=1, ls=":", label="ATE")
ax.set_xlabel("Customers (sorted by predicted effect)")
ax.set_ylabel("Estimated uplift on visit")
ax.set_title("Causal forest effect intervals (40 sampled customers)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
from packages.plotting import save_fig
save_fig(fig, "day3_forest_effect_intervals", "Causal forest 95% effect intervals per customer")
plt.show()

![Causal forest effect intervals](../../docs/figures/day3_forest_effect_intervals.png)

**Reading §3.** The mean interval width is *much* larger than the ATE, so only
a minority of customers have an interval that clears zero — while the
**majority** of point estimates are still positive. Both facts at once mean:
the forest thinks almost everyone is helped *slightly*, but it can only
*certify* a fraction of them individually. Take the consequence seriously on
Day 4: any per-customer profit claim built on these effects inherits this
uncertainty, so the optimizer must report profit **with** intervals rather
than as a point number.

## 4. CATE calibration — do predicted magnitudes match observed uplift?

Individual effects are never observed, so we cannot score one customer.
Instead: bin customers by **predicted** uplift (deciles), then compare the mean
prediction against the **observed** treated−control outcome difference in each
bin. The right shape is monotonic and near the diagonal.

This is a *different* question from Qini. Qini asks "does the **ranking**
work?"; this asks "do the **numbers** work?". A model can pass the first and
fail this — which is exactly the pattern to look for.

In [ ]:
calib = {}
for nm in ("forest", "dr", "x"):
    calib[nm] = cate_calibration_by_decile(ytef, scores[nm], ttef, n_bins=10)

pd.DataFrame({
    nm: {
        "spearman": tab.attrs["spearman_pred_vs_obs"],
        "pred_d1": tab["predicted_uplift"].iloc[0],
        "pred_d10": tab["predicted_uplift"].iloc[-1],
        "obs_d1": tab["observed_uplift"].iloc[0],
        "obs_d10": tab["observed_uplift"].iloc[-1],
    }
    for nm, tab in calib.items()
}).T.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colours = {"forest": "#64b5cd", "dr": "#ccb974", "x": "#8172b3"}
for nm, tab in calib.items():
    ax.plot(tab["predicted_uplift"], tab["observed_uplift"], "o-", label=nm, color=colours[nm])
# Reference: perfect calibration and the ATE.
lo = min(t["predicted_uplift"].min() for t in calib.values())
hi = max(t["predicted_uplift"].max() for t in calib.values())
ax.plot([lo, hi], [lo, hi], ls="--", lw=1, color="#333333", label="perfect")
ate = ytef[ttef == 1].mean() - ytef[ttef == 0].mean()
ax.axhline(ate, ls=":", lw=1, color="#c44e52", label=f"ATE ({ate:.3f})")
ax.set_xlabel("Predicted uplift (decile mean)")
ax.set_ylabel("Observed uplift (treated − control)")
ax.set_title("CATE calibration by decile — predicted vs observed", fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(
    fig, "day3_cate_calibration_deciles",
    "CATE calibration: predicted vs observed uplift by decile",
)
plt.show()

![CATE calibration by decile](../../docs/figures/day3_cate_calibration_deciles.png)

**Reading §4 — the central calibration finding.** The predictions spread
**wide** (the x-learner's top decile predicts roughly 0.21), while the
observed uplift per decile stays **bunched near the ATE** and moves noisily.
The rank correlation is positive but only moderate, and no model's observed
column is monotonic.

Two consequences, both load-bearing for Day 4:

- The models **rank** better than their **magnitudes** deserve — the ordering
  carries signal, the effect *sizes* are over-dispersed (over-confident).
- Therefore **never multiply a raw predicted uplift by a dollar value** to get
  profit. Recalibrate first (§5), or better, calibrate *and* validate on the
  validation split. This is precisely why the champion is chosen on Qini
  **and** profit, never on a magnitude metric alone.

## 5. Outcome-model calibration (per arm) and the recalibration decision

A T-learner's uplift is the **difference of two predicted probabilities**, and
the profit formula multiplies that difference by value-per-conversion. So a
miscalibrated probability distorts every profit figure even when the ranking
is perfect.

We score each arm's outcome model on **its own arm** with a **Brier score**
(mean squared error of the probability) plus its **skill vs the base rate** —
the floor being "predict the constant base rate for everyone". Then we try
**isotonic** and **Platt** recalibration, fitted on **validation only**.

In [ ]:
from lightgbm import LGBMClassifier

def make_clf():
    return LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, random_state=RANDOM_STATE, verbose=-1,
    )

# The T-learner's two outcome models, fitted per arm.
m_treat = make_clf().fit(Xv[tv == 1], yv[tv == 1])
m_ctrl = make_clf().fit(Xv[tv == 0], yv[tv == 0])

# Score each on its OWN arm (how each was trained).
p_tr, y_tr_own = m_treat.predict_proba(Xtef[ttef == 1])[:, 1], ytef[ttef == 1]
p_co, y_co_own = m_ctrl.predict_proba(Xtef[ttef == 0])[:, 1], ytef[ttef == 0]

def skill(y, p):
    brier = ((y - p) ** 2).mean()
    base = ((y - y.mean()) ** 2).mean()
    return brier, 1 - brier / base

rows = []
for nm, yy, pp in [("treated model", y_tr_own, p_tr), ("control model", y_co_own, p_co)]:
    brier, sk = skill(yy, pp)
    rows.append({"model": nm, "n": len(yy), "base_rate": round(yy.mean(), 4),
                 "brier": round(brier, 5), "skill_vs_base": round(sk, 4)})
print("Outcome-model calibration on test (own arm):")
print(pd.DataFrame(rows).set_index("model").to_string())

# Reliability: how far each bin sits off the diagonal.
for nm, yy, pp in [("treated model", y_tr_own, p_tr), ("control model", y_co_own, p_co)]:
    rel = reliability_by_arm(yy, pp, np.ones(len(yy)))
    gap = (rel["mean_predicted"] - rel["mean_observed"]).abs()
    print(
        f"  {nm} reliability: bins={len(rel)}"
        f"  mean|gap|={gap.mean():.4f}  max|gap|={gap.max():.4f}"
    )

In [ ]:
# Recalibration: fit on VALIDATION, score on TEST (never fit on test).
y_val_own = yvalf[tvalf == 1]
p_val_own = m_treat.predict_proba(Xvalf[tvalf == 1])[:, 1]
raw_brier = ((y_tr_own - p_tr) ** 2).mean()

print("RECALIBRATION (treated model, fit on val -> score on test)")
print(f"  raw     Brier: {raw_brier:.5f}")
for meth in ("isotonic", "platt"):
    fn = fit_recalibrator(p_val_own, y_val_own, method=meth)
    rb = ((y_tr_own - fn(p_tr)) ** 2).mean()
    verdict = "BETTER" if rb < raw_brier else "WORSE"
    print(f"  {meth:<8s} Brier: {rb:.5f}  (delta {rb - raw_brier:+.5f}, {verdict})")

**Reading §5 — the recalibration decision.** Both outcome models sit at a
slightly **negative** skill vs the constant base rate: on Brier, the features
barely beat predicting one number for everyone. That is the weak-heterogeneity
story again, now on the probability scale. Reliability gaps are moderate
(≈0.03–0.04 mean), so the probabilities are usable but not sharp.

On recalibration: **both methods improve the held-out Brier, with Platt
edging out isotonic.** The decision:

- **Apply Platt scaling per arm, fitted on validation only**, to any model
  whose scores feed a profit formula. It is monotone, stable on modest
  validation sets, and costs nothing.
- Keep **raw scores for ranking** — recalibration is monotone, so it cannot
  improve Qini, and we do not touch the ranking.
- Note the gain is *modest*: recalibration tidies the scale, it does not
  rescue the underlying signal. The recommendation for Day 4 stands: calibrate
  before valuing, but choose the model on Qini and profit.

## 6. The `spend` challenger — regression CATE on real dollars

`visit` is a coarse proxy. A customer convinced to spend real dollars matters
even if they would have visited anyway, so the Day-3 plan calls for a DR-learner
(and forest) on **`spend`** as a regression CATE challenger.

Two things differ from `visit`:

- The outcome is **continuous** (99% zeros), so `make_learner(...,
  target="spend")` switches every base learner to a regressor and the split
  stratifies on quantile bins of spend.
- Qini AUC / AUUC are **undefined** here (scikit-uplift requires a binary
  outcome), so the table reports only the direct **incremental outcome at k**
  in dollars — the treated−control mean difference among the top k.

In [ ]:
Xs, ys, ts = build_features(df, target="spend")
spend_splits, _ = encode_splits(make_splits(Xs, ys, ts, random_state=RANDOM_STATE))
Xs_tr, ys_tr, ts_tr = spend_splits["train"]
Xs_te, ys_te, ts_te = spend_splits["test"]
Xsv, tsv, ysv = Xs_tr.values.astype(float), ts_tr.values.astype(float), ys_tr.values.astype(float)
Xs_tef, ys_tef, ts_tef = (
    Xs_te.values.astype(float), ys_te.values.astype(float), ts_te.values.astype(float)
)

print(f"spend ATE on test: ${ys_tef[ts_tef == 1].mean() - ys_tef[ts_tef == 0].mean():.3f}")
print(f"converters in test: {(ys_tef > 0).sum()} of {len(ys_tef):,}")

spend_scores = {}
for nm in ("dr", "forest", "t", "x"):
    m = make_learner(nm, target="spend", random_state=RANDOM_STATE)
    m.fit(Xsv, tsv, ysv)
    spend_scores[nm] = m.predict(Xs_tef)
spend_scores["random"] = random_uplift(ys_tef, ts_tef, random_state=RANDOM_STATE)

print("\nSPEND-target ranking (incremental outcome at k, USD):")
print(ranking_table(ys_tef, ts_tef, spend_scores).to_string())

# NOISE FLOOR: a *random* ranking re-drawn over 30 fresh splits, for both
# targets. Spend has ~1% converters, so one split's uplift@k swings hugely;
# this distribution is the bar every single-split spend number above must clear.
rng_floor = np.random.default_rng(7)
floor_rows = []
for tgt_, Xt_, yt_, tt_ in (("spend", Xs, ys, ts), ("visit", X, y, t)):
    for seed_ in range(30):
        sp_, _ = encode_splits(make_splits(Xt_, yt_, tt_, random_state=seed_))
        _, ye_, te_ = sp_["test"]
        rand_ = rng_floor.random(len(ye_))
        for k_ in (0.1, 0.3):
            floor_rows.append(
                {
                    "target": tgt_,
                    "k": k_,
                    "uplift": uplift_at_k(
                        ye_.to_numpy(float), rand_, te_.to_numpy(float), k=k_
                    ),
                }
            )
floor = (
    pd.DataFrame(floor_rows)
    .groupby(["target", "k"])["uplift"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
print("\nRANDOM-ranking noise floor across 30 splits:")
floor

**Reading §6 — the spend findings.** Numbers below are single-split
(`RANDOM_STATE = 42`): uplift@10% / @30% — forest 1.87 / 0.88, DR 1.51 / 0.60,
T 0.57 / 0.64, random 0.23 / 0.70, X 0.07 / 0.09; test-split ATE ≈ $0.96
(full `mens` arm: $0.77, notebook 02).

- **The table's ordering does not survive its own noise floor.** A *random*
  ranking on spend spans **−0.89 to +4.18 at 10%** across 30 fresh splits
  (mean 1.07, **std 1.01**) and 0.07–1.88 at 30% (0.75 ± 0.47). The forest's
  1.87 and DR's 1.51 sit *inside* that band, and so does this seed's random
  draw of 0.23. One test split contains only 83 converters — on this metric a
  single split cannot separate models.
- **What survives the floor.** The forest and DR are *directionally* above the
  random mean at 10% (+0.8σ, +0.4σ), consistent with a real but shallow dollar
  signal; the **X-learner is the only model below random at both depths**
  (≈1σ at 10%, ≈1.4σ at 30%), which matches the structural story — its
  imputation step amplifies noise when 99% of outcomes are zero. The forest's
  own dilution (1.87 at 10% → 0.88 at 30%, back toward the ATE) is the
  classic pattern of a signal concentrated at the very top.
- **Why `visit` is the benchmark target:** the same random floor on `visit` is
  0.0759 ± 0.0113 at 30% — about **40× tighter** than spend. The 10-seed
  benchmark in §7 rests on a stable metric; nothing in this table does.

Stated plainly: §7 repeats **visit**, not spend — no multi-seed spend ranking
was run. Treat every dollar ordering above as suggestive only; Day 4's profit
work must carry this uncertainty with it (floor computed live in §6, and
mirrored in the [`docs/concepts.md`](../../docs/concepts.md) ledger, Day 3).


## 7. Stability — the 10-seed benchmark

One split can flatter a model. Notebook 03's headline gap (X-learner 0.0236 vs
response 0.0199) is *smaller than* typical single-split noise, so this section
repeats the whole train/score/evaluate loop over **10 seeds** and reports
**mean ± spread** — plus **profit at 30%** under the decided assumptions
(`configs/business_assumptions.yaml`), because the champion is chosen on Qini
**and** profit.

**Profit is paid in conversion dollars, not visits.** Models rank on
`visit` (the Day-2 target), but `profit_at_k` is evaluated on test-set
`conversion` — the fold returns it as a 7th array, aligned by row index.
Valuing visits at value-per-conversion would overstate profit by ~10x (Day 1
measured visits at ~11x the conversion rate).

The `make_fold` callable is injected, so `run_benchmark` knows nothing about
Hillstrom specifically.

In [ ]:
assumptions = BusinessAssumptions.from_yaml()
print("assumptions:", assumptions)

def make_fold(seed: int):
    """A fresh stratified split per seed; one-hot fit on that seed's train.

    Returns a 7th array — test-set **conversions** aligned by row index — so
    the benchmark can rank on visits but pay out in conversion dollars.
    Valuing visits at value-per-conversion would overstate profit ~10x.
    """
    raw = make_splits(X, y, t, random_state=seed)
    test_idx = raw["test"][0].index  # row labels before one-hot encoding
    sp, _ = encode_splits(raw)
    X_tr_, y_tr_, t_tr_ = sp["train"]
    X_te_, y_te_, t_te_ = sp["test"]
    conv_te = df.loc[test_idx, "conversion"].to_numpy(dtype=float)
    return (X_tr_.values.astype(float), t_tr_.values.astype(float), y_tr_.values.astype(float),
            X_te_.values.astype(float), t_te_.values.astype(float), y_te_.values.astype(float),
            conv_te)

result = run_benchmark(make_fold, target="visit", seeds=tuple(range(10)),
                       k=0.3, assumptions=assumptions)
result.summary

In [ ]:
# Spread across seeds: is any edge larger than the noise around it?
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in [
    (axes[0], "qini_auc", "Qini AUC across 10 seeds"),
    (axes[1], "profit_at_k", f"Profit at k={result.k:.0%} across 10 seeds"),
]:
    means = result.raw.groupby("model")[metric].mean().sort_values()
    stds = result.raw.groupby("model")[metric].std().reindex(means.index)
    ax.barh(means.index, means.values, xerr=stds.values, color="#4c72b0", alpha=0.85, capsize=3)
    ax.set_title(title, fontsize=11)
    ax.axvline(0, color="#999999", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "day3_benchmark_spread", "Mean +/- std across 10 seeds for Qini AUC and profit")
plt.show()

![Benchmark spread across 10 seeds](../../docs/figures/day3_benchmark_spread.png)

In [ ]:
# Does the top model's edge over second place exceed its own spread?
result.winners()

## 8. The champion — and why

**Selection rule (fixed in advance by the project rule: Qini AUC *and*
incremental profit, never Brier).** A model is declared champion only if its
mean Qini edge over second place exceeds its own standard deviation across
seeds; otherwise the verdict is a **statistical tie**, broken toward profit and
parsimony. Brier (§5) is calibration hygiene — it does not select.

**The result — 10 seeds, mean ± std, profit valued on conversions**
(per test split, n = 8,523; 2,557 targeted at k = 30%; full table in the
`winners()` output above and in [`docs/concepts.md`](../../docs/concepts.md)):

| Model | Qini AUC | uplift@30% (visit) | incr. conv @30% | profit @30% | $/contact |
| --- | --- | --- | --- | --- | --- |
| **Response (champion)** | **0.0231 ± 0.0094** | **0.0912** | **23.6** | **$569 ± 239** | **$0.223** |
| DR-learner | 0.0183 ± 0.0091 | 0.0911 | 20.7 | $466 ± 221 | $0.182 |
| S-learner | 0.0128 ± 0.0144 | 0.0854 | 22.4 | $524 ± 268 | $0.205 |
| X-learner | 0.0046 ± 0.0161 | 0.0849 | 21.4 | $491 ± 345 | $0.192 |
| T-learner | 0.0041 ± 0.0132 | 0.0857 | 20.6 | $463 ± 317 | $0.181 |
| Causal forest | 0.0027 ± 0.0182 | 0.0767 | 21.7 | $500 ± 304 | $0.196 |
| *Uniform (Day-1 ATE)* | *≈ 0 (random)* | *0.0766* | *17.4* | *≈ $351* | *$0.137* |

**Verdict: statistical tie at the top → the tie-break names the response
model.** Response's Qini edge over DR (0.0048) is *smaller than its own std*
(0.0094) — even by the gentler standard-error test it is only ~1.2 SE, not
separable. On the profit tie-break response leads as well ($569 vs S's $524,
both within one std), and it is one model rather than two. So the champion of
Day 3 is the **response baseline**: on `visit`, **no uplift model clears it.**

**What the table settles:**

1. **Day-2's X-learner "lead" was noise.** 0.0236 on one split becomes
   0.0046 ± 0.0161 over seeds — 0.9 SE above a random ranking. X, T and the
   forest are all *statistically at random* on `visit`; only response, DR and
   S clear the zero line (7.8, 6.4 and 2.8 SE respectively).
2. **Qini and profit disagree on the ordering.** DR is 2nd on Qini but 5th on
   profit; S is 3rd on Qini but 2nd on profit. Ranking visits is not counting
   dollars — the reason §7 values profit on conversions and the project rule
   demands both metrics.
3. **Every model beats uniform targeting, the response model most of all.**
   $0.223/contact vs Day-1's uniform $0.137 (+62%); 23.6 vs 17.4 incremental
   conversions at 30% (9.2 vs 6.8 per 1,000 contacted); ~2.9 standard errors
   of the mean above the uniform profit line. The hoped-for reversal — profit
   punishing the response model's "sure things" — **did not materialise at
   30% depth**; if it appears anywhere it will be in Day 4's deeper curves.

**For Day 4:** the model field is flat, so the decision layer (depth choice,
segments, policy tree) is where any real gain must come from — and the spend
side keeps its §6 caveat: single-split dollars, no multi-seed spend claim.
